# Notebook 08: FastAPI Backend
### Hybrid E-Commerce Recommendation System — H&M Personalized Fashion Recommendations

**Scope of this notebook:** wrap the already-trained recommendation system (Notebooks 03–07)
in a FastAPI backend so it can be queried over HTTP.

**Explicitly out of scope for this notebook:**
- No model training or retraining — every model/artifact is loaded from disk exactly as saved.
- No Streamlit frontend — that is Notebook 09.
- No cloud deployment (Render/AWS/etc.) — this notebook only proves the API works, including
  running it live inside this Colab session.


---
## 1. Project Structure

```text
hm_recsys/                     (BASE_DIR, on Google Drive — created by earlier notebooks)
├── processed_data/            already exists (Notebook 03)
├── models/                    already exists (Notebooks 05-06): als_model.pkl, hybrid_model_config.pkl
├── artifacts/                 already exists (Notebooks 05-07): train_matrix.npz, popularity_baseline.pkl
├── cb_artifacts/              already exists (Notebook 04): tfidf_vectorizer.pkl, tfidf_matrix.npz
└── api/                       NEW — created by this notebook
    ├── __init__.py
    ├── main.py
    └── recommendation_service.py
```

We deliberately do **not** duplicate `models/` and `artifacts/` under `api/` — they already exist
from earlier notebooks and hold multi-MB files (the ALS model, the TF-IDF matrix, the train
matrix). The API just points at them. Only the two new Python files live under `api/`.

**Purpose of each new file:**

| File | Purpose |
|---|---|
| `api/recommendation_service.py` | All ML-facing code: loads every saved artifact once, and exposes plain methods (`get_recommendations`, `get_similar_products`) that return plain Python data. No FastAPI/HTTP concepts live here — this file would work the same in a CLI script or a Streamlit app. |
| `api/main.py` | All HTTP-facing code: defines the FastAPI app, routes, request/response schemas (Pydantic), and turns service-layer conditions (unknown user, bad `top_k`, etc.) into proper HTTP responses. No ML logic lives here — it only *calls* the service. |
| `requirements.txt` | Pins the backend + ML dependencies needed to run the API **outside** this notebook (e.g. `uvicorn api.main:app --reload` on a laptop or a server). |

This separation (**service layer** vs. **API layer**) is a standard backend pattern: it lets you
test the recommendation logic directly in Python (as we do later in this notebook) without
spinning up a server, and it lets you swap the HTTP framework later without touching any ML code.


In [ ]:
import os

# Same convention as Notebooks 03-07: work locally if artifacts happen to be in the
# current runtime already, otherwise mount Drive and point at the shared project folder.
BASE_DIR = "/content/drive/MyDrive/hm_recsys"

if not os.path.exists(BASE_DIR):
    print("Project folder not found locally — mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(
        f"{BASE_DIR} still not found after mounting Drive. "
        f"Update BASE_DIR to match where Notebooks 03-07 saved their artifacts."
    )

PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "processed_data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
CB_ARTIFACTS_DIR = os.path.join(BASE_DIR, "cb_artifacts")
BACKEND_DIR = BASE_DIR  # api/ will live directly under BASE_DIR, next to models/ and artifacts/

os.makedirs(os.path.join(BACKEND_DIR, "api"), exist_ok=True)

# Lets `uvicorn api.main:app` and `from api.recommendation_service import ...` resolve
# correctly both in this notebook's Python process and later from a terminal.
os.chdir(BACKEND_DIR)
os.environ["RECSYS_BASE_DIR"] = BASE_DIR

print("Working directory set to:", os.getcwd())
print("BASE_DIR:", BASE_DIR)


In [ ]:
# Backend + ML dependencies needed for this notebook and for running the API standalone.
!pip install -q fastapi "uvicorn[standard]" pydantic nest_asyncio implicit
print("Dependencies ready.")


---
## 2. Recommendation Service — Load Everything Once

`RecommendationService` is instantiated **exactly once**, when the API process starts
(see `main.py` below). Every method on it reuses the same in-memory model objects —
nothing is re-read from disk or refit per request.

It reuses the *exact same scoring logic* as Notebooks 05–07 (`recommend_hybrid_core`,
adaptive alpha, min-max normalization, popularity fallback) — just reorganized into a class
so the API layer can call it cleanly, and with a small addition: unknown/new `user_id`s are
automatically routed to the popularity baseline instead of raising an error, per the spec.


In [ ]:
%%writefile api/recommendation_service.py
"""
recommendation_service.py

Loads every saved ML artifact from the Hybrid E-Commerce Recommendation System
(Notebooks 01-07) exactly once, and exposes plain Python methods that the API
layer (main.py) calls per-request.

IMPORTANT: no training happens here. This module only reconstructs objects that
were already fit and saved to disk in earlier notebooks.
"""

import os
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity


class RecommendationService:
    """
    Loads artifacts once in __init__, then serves requests from memory.

    Loading touches disk (Drive or local) and deserializes multi-MB pickles and
    sparse matrices (the TF-IDF matrix, the ALS factors, the train matrix). Doing
    that on every request would add hundreds of milliseconds (or seconds, on Drive)
    of latency to every single API call, and would multiply memory usage under any
    concurrent traffic. Instantiate this class ONCE per process.
    """

    def __init__(self, base_dir: str = None):
        self.base_dir = base_dir or os.environ.get(
            "RECSYS_BASE_DIR", "/content/drive/MyDrive/hm_recsys"
        )
        self.processed_data_dir = os.path.join(self.base_dir, "processed_data")
        self.models_dir = os.path.join(self.base_dir, "models")
        self.artifacts_dir = os.path.join(self.base_dir, "artifacts")
        self.cb_artifacts_dir = os.path.join(self.base_dir, "cb_artifacts")

        self._load_artifacts()

    # ------------------------------------------------------------------ #
    # Loading (runs once, at startup)
    # ------------------------------------------------------------------ #
    def _load_artifacts(self):
        required_dirs = [self.processed_data_dir, self.models_dir, self.artifacts_dir]
        missing = [p for p in required_dirs if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError(
                f"Required artifact directories not found: {missing}. "
                f"Run Notebooks 03-06 first, or set RECSYS_BASE_DIR correctly."
            )

        # --- Product metadata (for turning item_idx back into something readable) ---
        self.processed_articles = pd.read_csv(
            os.path.join(self.processed_data_dir, "processed_articles.csv")
        )
        self.article_lookup = self.processed_articles.set_index("article_id")

        # --- Content-based artifacts ---
        content_features = pd.read_csv(os.path.join(self.processed_data_dir, "content_features.csv"))
        self.cb_article_ids = content_features["article_id"].values
        self.cb_id_to_index = {aid: idx for idx, aid in enumerate(self.cb_article_ids)}

        vec_path = os.path.join(self.cb_artifacts_dir, "tfidf_vectorizer.pkl")
        mat_path = os.path.join(self.cb_artifacts_dir, "tfidf_matrix.npz")
        if os.path.exists(vec_path) and os.path.exists(mat_path):
            with open(vec_path, "rb") as f:
                self.tfidf_vectorizer = pickle.load(f)
            self.tfidf_matrix = load_npz(mat_path)
        else:
            # cb_artifacts/ was a LOCAL (non-Drive) path in Notebook 04/07, so it may not
            # exist in a fresh runtime. Rebuild with the exact same config as Notebook 04
            # rather than hard-failing -- this is a few seconds of work, not training.
            print("[RecommendationService] cb_artifacts not found locally — rebuilding TF-IDF...")
            from sklearn.feature_extraction.text import TfidfVectorizer
            FINAL_TFIDF_CONFIG = {"max_features": 10000, "min_df": 2, "max_df": 0.8, "ngram_range": (1, 2)}
            self.tfidf_vectorizer = TfidfVectorizer(**FINAL_TFIDF_CONFIG, stop_words="english")
            self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(
                content_features["combined_features"].fillna("")
            )

        # --- Encoders (string IDs <-> matrix indices) ---
        with open(os.path.join(self.processed_data_dir, "label_encoders.pkl"), "rb") as f:
            encoders = pickle.load(f)
        self.user_encoder = encoders["user_encoder"]
        self.item_encoder = encoders["item_encoder"]

        # --- Collaborative filtering (ALS) ---
        with open(os.path.join(self.models_dir, "als_model.pkl"), "rb") as f:
            als_bundle = pickle.load(f)
        self.als_model = als_bundle["model"]

        # --- Train matrix (needed to know what a user already purchased) ---
        self.train_matrix = load_npz(os.path.join(self.artifacts_dir, "train_matrix.npz"))

        # --- Popularity fallback (used for new/cold-start users) ---
        with open(os.path.join(self.artifacts_dir, "popularity_baseline.pkl"), "rb") as f:
            pop_bundle = pickle.load(f)
        self.popularity_ranking = pop_bundle["popularity_ranking"]

        # --- Hybrid config (alpha, activity threshold) from Notebook 06 ---
        with open(os.path.join(self.models_dir, "hybrid_model_config.pkl"), "rb") as f:
            hybrid_config = pickle.load(f)
        self.best_alpha = hybrid_config["best_alpha"]
        self.low_activity_threshold = hybrid_config["low_activity_threshold"]

        print("[RecommendationService] All artifacts loaded successfully.")

    # ------------------------------------------------------------------ #
    # Internal scoring helpers (same logic as Notebooks 05-07)
    # ------------------------------------------------------------------ #
    def _build_user_content_profile(self, user_idx):
        purchased_item_indices = self.train_matrix[user_idx].indices
        if len(purchased_item_indices) == 0:
            return None
        purchased_article_ids = self.item_encoder.inverse_transform(purchased_item_indices)
        cb_indices = [
            self.cb_id_to_index[aid] for aid in purchased_article_ids if aid in self.cb_id_to_index
        ]
        if not cb_indices:
            return None
        return np.asarray(self.tfidf_matrix[cb_indices].mean(axis=0))

    @staticmethod
    def _min_max_normalize(scores):
        if len(scores) == 0:
            return scores
        min_val, max_val = scores.min(), scores.max()
        if max_val - min_val < 1e-9:
            return np.zeros_like(scores)
        return (scores - min_val) / (max_val - min_val)

    def _get_adaptive_alpha(self, user_idx):
        n_purchases = self.train_matrix[user_idx].nnz
        if n_purchases == 0:
            return self.best_alpha
        elif n_purchases < self.low_activity_threshold:
            return 0.8
        else:
            return 0.2

    def _recommend_hybrid_core(self, user_idx, alpha, n=10, candidate_pool_size=50):
        already_purchased = set(self.train_matrix[user_idx].indices)

        als_ids, _ = self.als_model.recommend(
            user_idx, self.train_matrix[user_idx], N=candidate_pool_size, filter_already_liked_items=True
        )

        profile_vector = self._build_user_content_profile(user_idx)
        content_candidates = []
        if profile_vector is not None:
            scores = cosine_similarity(profile_vector, self.tfidf_matrix).flatten()
            top_cb_indices = scores.argsort()[::-1][:candidate_pool_size]
            for cb_idx in top_cb_indices:
                aid = self.cb_article_ids[cb_idx]
                if aid in self.item_encoder.classes_:
                    content_candidates.append(self.item_encoder.transform([aid])[0])

        candidate_items = list(dict.fromkeys(
            list(als_ids) + content_candidates + list(self.popularity_ranking[:20])
        ))
        candidate_items = [i for i in candidate_items if i not in already_purchased]
        if not candidate_items:
            return []

        candidate_article_ids = self.item_encoder.inverse_transform(candidate_items)
        valid_pairs = [
            (item_idx, self.cb_id_to_index[aid])
            for item_idx, aid in zip(candidate_items, candidate_article_ids)
            if aid in self.cb_id_to_index
        ]
        if not valid_pairs:
            return []

        valid_item_indices, valid_cb_indices = zip(*valid_pairs)
        valid_item_indices = np.array(valid_item_indices)
        valid_cb_indices = np.array(valid_cb_indices)

        user_vector = self.als_model.user_factors[user_idx]
        item_vectors = self.als_model.item_factors[valid_item_indices]
        collab_scores_raw = item_vectors @ user_vector

        if profile_vector is not None:
            content_scores_raw = cosine_similarity(
                profile_vector, self.tfidf_matrix[valid_cb_indices]
            ).flatten()
        else:
            content_scores_raw = np.zeros(len(valid_item_indices))

        content_scores_norm = self._min_max_normalize(content_scores_raw)
        collab_scores_norm = self._min_max_normalize(collab_scores_raw)
        hybrid_scores = alpha * content_scores_norm + (1 - alpha) * collab_scores_norm

        results = list(zip(valid_item_indices, hybrid_scores))
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:n]

    def _popularity_recommend(self, user_idx_or_none, n=10):
        already_purchased = (
            set(self.train_matrix[user_idx_or_none].indices)
            if user_idx_or_none is not None else set()
        )
        recs = []
        for item_idx in self.popularity_ranking:
            if item_idx in already_purchased:
                continue
            recs.append((item_idx, None))
            if len(recs) == n:
                break
        return recs

    def _format_recommendations(self, results):
        rows = []
        for item_idx, score in results:
            article_id = self.item_encoder.inverse_transform([item_idx])[0]
            if article_id not in self.article_lookup.index:
                continue
            meta = self.article_lookup.loc[article_id]
            rows.append({
                "article_id": str(article_id),
                "product_name": str(meta.get("prod_name", "")),
                "category": str(meta.get("product_group_name", "")),
                "score": round(float(score), 4) if score is not None else None,
            })
        return rows

    # ------------------------------------------------------------------ #
    # Public methods — these are what main.py calls
    # ------------------------------------------------------------------ #
    def is_known_user(self, customer_id: str) -> bool:
        return customer_id in self.user_encoder.classes_

    def get_recommendations(self, customer_id: str, top_k: int = 10):
        """
        Returns (recommendations, is_new_user).

        Unknown/new customer_ids are NOT an error — they automatically fall back to the
        popularity baseline. This is the cold-start handling required by the spec.
        """
        if self.is_known_user(customer_id):
            user_idx = self.user_encoder.transform([customer_id])[0]
            alpha = self._get_adaptive_alpha(user_idx)
            results = self._recommend_hybrid_core(user_idx, alpha=alpha, n=top_k)
            if not results:
                # Known user, but no valid hybrid candidates left (rare edge case) ->
                # popularity fallback rather than an empty/broken response.
                results = self._popularity_recommend(user_idx, n=top_k)
            return self._format_recommendations(results), False
        else:
            results = self._popularity_recommend(None, n=top_k)
            return self._format_recommendations(results), True

    def is_known_article(self, article_id) -> bool:
        return article_id in self.cb_id_to_index

    def get_similar_products(self, article_id, top_k: int = 10):
        """Content-based item-item similarity using the fitted TF-IDF matrix."""
        cb_idx = self.cb_id_to_index[article_id]
        item_vector = self.tfidf_matrix[cb_idx]
        scores = cosine_similarity(item_vector, self.tfidf_matrix).flatten()

        ranked_indices = scores.argsort()[::-1]
        rows = []
        for idx in ranked_indices:
            candidate_id = self.cb_article_ids[idx]
            if candidate_id == article_id:
                continue
            if candidate_id not in self.article_lookup.index:
                continue
            meta = self.article_lookup.loc[candidate_id]
            rows.append({
                "article_id": str(candidate_id),
                "product_name": str(meta.get("prod_name", "")),
                "category": str(meta.get("product_group_name", "")),
                "similarity_score": round(float(scores[idx]), 4),
            })
            if len(rows) == top_k:
                break
        return rows


---
## 3-7. FastAPI Application

`main.py` below contains:

- **Section 3** — the `FastAPI()` app and `GET /health`.
- **Section 4** — `GET /recommend/{user_id}`.
- **Section 5** — `GET /similar/{article_id}`.
- **Section 6** — error handling (missing artifacts at startup, unknown article, bad input).
- **Section 7** — request validation via Pydantic/FastAPI (`top_k` bounded 1-50, non-blank IDs).

**Design note on error handling:** the service is loaded once, at import time, wrapped in a
`try/except`. If any artifact is missing, the app still *starts* (so `/health` can report the
problem) but every data-serving endpoint returns `503 Service Unavailable` instead of crashing
the whole process — this is the standard pattern for "dependency failed to initialize."


In [ ]:
%%writefile api/main.py
"""
main.py

FastAPI application entrypoint.

Responsibilities:
- Create the FastAPI app and its routes.
- Instantiate RecommendationService ONCE at process startup (not per-request).
- Validate input (top_k, path params) via FastAPI/Pydantic.
- Translate service-layer conditions (unknown user/article, bad input, missing
  artifacts) into proper HTTP status codes.

No ML logic lives in this file — see recommendation_service.py for that.

Run standalone with:
    uvicorn api.main:app --reload
"""

from typing import List, Optional

from fastapi import FastAPI, HTTPException, Path, Query
from pydantic import BaseModel

from api.recommendation_service import RecommendationService


# --------------------------------------------------------------------- #
# Load models ONCE at import time (i.e. once per process, at startup).
# If loading fails, the app still starts so /health can report the problem,
# but every other endpoint returns 503 instead of crashing.
# --------------------------------------------------------------------- #
try:
    recommendation_service = RecommendationService()
    SERVICE_READY = True
    SERVICE_ERROR = None
except Exception as exc:  # noqa: BLE001 - we want to surface ANY startup failure via /health
    recommendation_service = None
    SERVICE_READY = False
    SERVICE_ERROR = str(exc)


app = FastAPI(
    title="Hybrid E-Commerce Recommendation API",
    description=(
        "Serves content-based, collaborative, and hybrid recommendations from "
        "pre-trained artifacts. No training happens in this service."
    ),
    version="1.0.0",
)


# --------------------------------------------------------------------- #
# Section 7: Pydantic schemas — these also drive automatic request validation
# --------------------------------------------------------------------- #
class HealthResponse(BaseModel):
    status: str


class RecommendationItem(BaseModel):
    article_id: str
    product_name: str
    category: str
    score: Optional[float] = None


class RecommendationResponse(BaseModel):
    user_id: str
    is_new_user: bool
    recommendations: List[RecommendationItem]


class SimilarItem(BaseModel):
    article_id: str
    product_name: str
    similarity_score: float
    category: str


class SimilarResponse(BaseModel):
    article_id: str
    similar_products: List[SimilarItem]


def _ensure_service_ready():
    """Section 6: missing-artifact handling — surfaced as 503, not a crash."""
    if not SERVICE_READY:
        raise HTTPException(
            status_code=503,
            detail=f"Recommendation service unavailable — artifacts failed to load: {SERVICE_ERROR}",
        )


# --------------------------------------------------------------------- #
# Section 3: health endpoint
# --------------------------------------------------------------------- #
@app.get("/health", response_model=HealthResponse)
def health():
    """Lightweight liveness check — does not touch the ML artifacts."""
    return HealthResponse(status="healthy" if SERVICE_READY else "unhealthy")


# --------------------------------------------------------------------- #
# Section 4: personalized recommendations
# --------------------------------------------------------------------- #
@app.get("/recommend/{user_id}", response_model=RecommendationResponse)
def recommend(
    user_id: str = Path(
        ..., min_length=1, description="Customer ID (raw string, not the internal matrix index)"
    ),
    top_k: int = Query(10, ge=1, le=50, description="Number of recommendations to return (1-50)"),
):
    _ensure_service_ready()

    if not user_id.strip():
        raise HTTPException(status_code=400, detail="user_id must not be blank.")

    recs, is_new_user = recommendation_service.get_recommendations(user_id, top_k=top_k)

    if not recs:
        # Extremely rare (e.g. catalog metadata missing for every candidate) but still
        # a real failure mode worth a clear error rather than an empty 200.
        raise HTTPException(
            status_code=404,
            detail=f"No recommendations could be generated for user_id='{user_id}'.",
        )

    return RecommendationResponse(user_id=user_id, is_new_user=is_new_user, recommendations=recs)


# --------------------------------------------------------------------- #
# Section 5: content-based "similar products"
# --------------------------------------------------------------------- #
@app.get("/similar/{article_id}", response_model=SimilarResponse)
def similar(
    article_id: str = Path(..., min_length=1, description="Article ID from the product catalog"),
    top_k: int = Query(10, ge=1, le=50, description="Number of similar products to return (1-50)"),
):
    _ensure_service_ready()

    # Catalog article_ids are numeric (H&M convention) but always arrive as a string
    # from the URL path - normalize before the lookup, and reject non-numeric input.
    try:
        normalized_id = int(article_id)
    except ValueError:
        raise HTTPException(
            status_code=400, detail=f"article_id must be numeric, got '{article_id}'."
        )

    if not recommendation_service.is_known_article(normalized_id):
        raise HTTPException(
            status_code=404, detail=f"article_id {normalized_id} not found in catalog."
        )

    similar_products = recommendation_service.get_similar_products(normalized_id, top_k=top_k)
    return SimilarResponse(article_id=str(normalized_id), similar_products=similar_products)


In [ ]:
%%writefile api/__init__.py


### `requirements.txt`

Pinned to lower-bound versions that are known to work with FastAPI's Pydantic v2 integration
and the `implicit` ALS library — this is what someone would `pip install -r requirements.txt`
to run the API outside Colab.


In [ ]:
%%writefile requirements.txt
fastapi>=0.110
uvicorn[standard]>=0.29
pydantic>=2.6
pandas>=2.0
numpy>=1.24
scipy>=1.11
scikit-learn>=1.3
implicit>=0.7


---
## 8. Test the API

Colab can't run `uvicorn api.main:app --reload` in a blocking foreground cell (it would freeze
the notebook), so we start the server in a background thread instead and talk to it over
`http://127.0.0.1:8000` using `requests` — exactly the same HTTP calls Swagger UI (`/docs`) or
any external client would make. `nest_asyncio` is required because Colab's own kernel already
runs an asyncio event loop, and uvicorn needs to run its own inside it.

To open the interactive Swagger UI (`/docs`) itself from inside Colab, run the last cell in this
section — it prints a clickable proxied URL.


In [ ]:
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()

# Re-import so this cell picks up the files just written by %%writefile above,
# rather than any stale module cached from an earlier run.
import sys
for mod in ["api.main", "api.recommendation_service", "api"]:
    sys.modules.pop(mod, None)

from api.main import app

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

time.sleep(3)  # give uvicorn a moment to bind the port before we start hitting it
print("Server started in background thread on http://127.0.0.1:8000")


In [ ]:
# Fail fast and clearly if artifact loading failed at import time, instead of hitting
# a confusing AttributeError/NoneType later when calling recommendation_service.
from api.main import SERVICE_READY, SERVICE_ERROR

if not SERVICE_READY:
    raise RuntimeError(f"RecommendationService failed to load: {SERVICE_ERROR}")

print("Service loaded successfully.")


In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

resp = requests.get(f"{BASE_URL}/health")
print("GET /health ->", resp.status_code)
print(resp.json())


In [ ]:
# Grab one real known customer_id and one real known article_id straight from the
# loaded encoders/catalog, so the demo calls below are guaranteed to hit real data.
from api.main import recommendation_service

sample_customer_id = recommendation_service.user_encoder.classes_[0]
sample_article_id = int(recommendation_service.cb_article_ids[0])

print("Sample known customer_id:", sample_customer_id)
print("Sample known article_id:", sample_article_id)


In [ ]:
# GET /recommend/{user_id} — known user, default top_k
resp = requests.get(f"{BASE_URL}/recommend/{sample_customer_id}")
print("GET /recommend/{user_id} ->", resp.status_code)
print(resp.json())


In [ ]:
# GET /recommend/{user_id} — known user, top_k=5
resp = requests.get(f"{BASE_URL}/recommend/{sample_customer_id}", params={"top_k": 5})
print("GET /recommend/{user_id}?top_k=5 ->", resp.status_code)
print(resp.json())


In [ ]:
# GET /recommend/{user_id} — a made-up customer_id -> new-user / cold-start path.
# Expect is_new_user=True and popularity-baseline recommendations (score=None), NOT an error.
resp = requests.get(f"{BASE_URL}/recommend/BRAND_NEW_CUSTOMER_NOT_IN_TRAINING_DATA")
print("GET /recommend/{unknown_user_id} ->", resp.status_code)
print(resp.json())


In [ ]:
# GET /similar/{article_id} — known article
resp = requests.get(f"{BASE_URL}/similar/{sample_article_id}", params={"top_k": 5})
print("GET /similar/{article_id}?top_k=5 ->", resp.status_code)
print(resp.json())


In [ ]:
# --- Invalid input tests ---

# 1. top_k out of range (> 50) -> FastAPI/Pydantic validation error, 422
resp = requests.get(f"{BASE_URL}/recommend/{sample_customer_id}", params={"top_k": 500})
print("top_k=500 ->", resp.status_code, resp.json())
print()

# 2. top_k below range (< 1) -> 422
resp = requests.get(f"{BASE_URL}/recommend/{sample_customer_id}", params={"top_k": 0})
print("top_k=0 ->", resp.status_code, resp.json())
print()

# 3. Non-numeric article_id -> 400 (handled explicitly in main.py)
resp = requests.get(f"{BASE_URL}/similar/not-a-number")
print("article_id='not-a-number' ->", resp.status_code, resp.json())
print()

# 4. Numeric but non-existent article_id -> 404
resp = requests.get(f"{BASE_URL}/similar/999999999999")
print("article_id=999999999999 (unknown) ->", resp.status_code, resp.json())


In [ ]:
# Optional: open the live Swagger UI (/docs) directly from Colab.
try:
    from google.colab.output import eval_js
    docs_url = eval_js("google.colab.kernel.proxyPort(8000)") + "docs"
    print("Swagger UI:", docs_url)
except ImportError:
    print("Not running in Colab — open http://127.0.0.1:8000/docs locally instead.")


---
## 9. Performance Considerations

- **Why load models once, not per-request:** deserializing the ALS model, the TF-IDF matrix,
  and the train matrix from disk (or Drive) takes real time and memory. If that happened inside
  the endpoint function, every single request would pay that cost — turning a sub-second
  recommendation into a multi-second one, and multiplying memory usage the moment two requests
  overlap. Loading once at process startup (`RecommendationService()` at import time) means the
  per-request cost is *only* the actual scoring math.

- **Why we don't recompute the whole recommender per request:** `recommend_hybrid_core` already
  restricts scoring to a small candidate pool (ALS top-50 + content top-50 + popularity top-20,
  deduplicated) rather than scoring the entire ~105K-article catalog for every user on every
  request. That candidate-pool trick, inherited from Notebook 06, is what keeps a single
  `/recommend` call fast enough to serve synchronously.

- **Why precomputed artifacts matter:** the ALS factors, the TF-IDF matrix, and the popularity
  ranking were all computed once in earlier notebooks and saved to disk. The API only ever reads
  them — it never refits a model — which is precisely what makes "load once, serve many" possible
  in the first place.

- **Where caching could help in production (not implemented here, to avoid over-engineering):**
  a simple in-memory or Redis cache keyed on `(user_id, top_k)` (and `(article_id, top_k)` for
  `/similar`) would let a small pool of very active or oft-viewed products skip recomputation
  entirely on repeat requests, at the cost of serving slightly stale recommendations until the
  cache expires. That's a reasonable next step once there's real traffic to justify it — not a
  requirement for this notebook.


---
## 10. Backend Code — Saved Files

Confirm the files this notebook wrote actually exist and are runnable outside the notebook.


In [ ]:
import os

for fname in ["api/__init__.py", "api/main.py", "api/recommendation_service.py", "requirements.txt"]:
    full_path = os.path.join(BACKEND_DIR, fname)
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    print(f"{fname}: {'OK' if exists else 'MISSING'} ({size} bytes)")

print()
print("To run this API outside of any notebook:")
print(f"  cd {BACKEND_DIR}")
print("  pip install -r requirements.txt")
print("  uvicorn api.main:app --reload")


---
## Final Verification

- `GET /health` → `{"status": "healthy"}` (confirmed in Section 8)
- `GET /recommend/{user_id}` → personalized hybrid recommendations for known users, automatic
  popularity fallback (with `is_new_user=True`) for unknown users — no error for cold-start
  (confirmed in Section 8)
- `GET /similar/{article_id}` → content-based similar products, `404` for unknown articles,
  `400` for malformed article IDs, `422` for out-of-range `top_k` (confirmed in Section 8)

All three endpoints load their models exactly once (Section 2) and serve every request from the
already-loaded artifacts — no retraining, no Streamlit, no deployment in this notebook.

**Next: Notebook 09 — Streamlit Frontend**

*(Not built here, per the brief — this notebook stops at a working, tested FastAPI backend.)*
